In [1]:
from three_cluster_box import tri_cluster_box
import pandas as pd
import numpy as np

In [2]:
requirements = pd.read_csv('Req_Res_Con_Tables/Requirements_filtered.csv', index_col = False)['Требование']
responsibilities = pd.read_csv('Req_Res_Con_Tables/Responsibilities_filtered.csv', index_col = False)['Обязанность']
conditions = pd.read_csv('Req_Res_Con_Tables/Conditions_filtered.csv', index_col = False)['Условие']

In [3]:
def run_clusterization(corelevance_matrix, strategy, params):
    new_box = tri_cluster_box(corelevance_matrix)

    # Развилка по сценариям
    if strategy == 'classic':
        clusters = new_box.make_tri_clusters_classical(*params)
    elif strategy == 'updated_matrix':
        clusters = new_box.make_tri_clusters_on_updated_matrix(*params)

    resulting_table = pd.DataFrame(columns = ['Требования', 'Обязанности', "Условия"])

    reqs, resp, cons = [], [], []

    for cluster in clusters:
        temp_reqs = list(cluster['dim_1'])
        temp_resp = list(cluster['dim_2'])
        temp_cons = list(cluster['dim_3'])

        reqs.append(requirements.loc[temp_reqs].tolist())
        resp.append(responsibilities.loc[temp_resp].tolist())
        cons.append(conditions.loc[temp_cons].tolist())

    resulting_table['Требования'] = reqs
    resulting_table['Обязанности'] = resp
    resulting_table['Условия'] = cons

    intensities, contributions = new_box.calculate_intensity_and_contribution(clusters)
    resulting_table['Вклады'] = contributions
    resulting_table['Интенсивности'] = intensities

    return resulting_table

### 1. Kulch

In [4]:
kulch_corelevance_0_30 = np.load('3D_relevance_matrices/kulch_corelevance_matrix_0_30.npy')

Updated matrix

In [5]:
results = run_clusterization(kulch_corelevance_0_30, 'updated_matrix', [0])
results.to_csv('kulch_updated.csv', index=False)

100%|██████████| 157/157 [00:07<00:00, 19.98it/s]


In [6]:
results = run_clusterization(kulch_corelevance_0_30, 'updated_matrix', [0, 10])
results.to_csv('kulch_updated_limit_size.csv', index=False)

100%|██████████| 157/157 [00:01<00:00, 110.60it/s]


Discard Something

In [24]:
results = run_clusterization(kulch_corelevance_0_30, 'classic', [0, False, True, False, False])
results.to_csv('kulch_discard_cond.csv', index=False)

100%|██████████| 150/150 [00:23<00:00,  6.51it/s]


In [9]:
results = run_clusterization(kulch_corelevance_0_30, 'classic', [0, False, False, True, False])
results.to_csv('kulch_discard_resp.csv', index=False)

  0%|          | 0/157 [00:00<?, ?it/s]

100%|██████████| 157/157 [00:17<00:00,  9.17it/s]


In [10]:
results = run_clusterization(kulch_corelevance_0_30, 'classic', [0, False, False, True, False, False, 30])
results.to_csv('kulch_discard_resp_size_limit.csv', index=False)

100%|██████████| 157/157 [00:07<00:00, 21.27it/s] 


In [11]:
results = run_clusterization(kulch_corelevance_0_30, 'classic', [0, False, False, True, False, False, 10])
results.to_csv('kulch_discard_resp_size_limit_10.csv', index=False)

100%|██████████| 157/157 [00:04<00:00, 32.45it/s] 


Cluster size limitation

In [6]:
results = run_clusterization(kulch_corelevance_0_30, 'updated_matrix', [0, 10])
results.to_csv('kulch_0_30_cluster_size_limit_10.csv', index=False)

  0%|          | 0/150 [00:00<?, ?it/s]

100%|██████████| 150/150 [00:01<00:00, 111.51it/s]


In [5]:
results = run_clusterization(kulch_corelevance_0_30, 'classic', [0, True, False, False, False, False, 5])
results.to_csv('kulch_0_30_cluster_size_limit_10_decrease_intensity.csv', index=False)

100%|██████████| 150/150 [3:56:12<00:00, 94.48s/it]   


In [10]:
results = run_clusterization(kulch_corelevance_0_30, 'classic', [0, 0, 0, 1, 0, 0, 5])
results.to_csv('kulch_0_30_cluster_size_limit_5_discard_responsibilities.csv', index=False)

100%|██████████| 150/150 [00:02<00:00, 74.40it/s] 


Decrease intensities

In [8]:
results = run_clusterization(kulch_corelevance_0_30, 'classic', [0, True])
results.to_csv('kulch_classical_decrease_intensities.csv', index=False)

  0%|          | 0/157 [05:11<?, ?it/s]


KeyboardInterrupt: 

### 2. No weighting strategy

In [11]:
no_weighting_corelevance_0_30 = np.load('no_weighting_corelevance_matrix_0_30.npy')

In [20]:
results_2 = run_clusterization(no_weighting_corelevance_0_30, 'classic', [0, False, True, False, False])
results_2.to_csv('discard_cond.csv', index=False)

  0%|          | 0/150 [00:00<?, ?it/s]

100%|██████████| 150/150 [00:12<00:00, 12.41it/s]


In [21]:

results_3 = run_clusterization(no_weighting_corelevance_0_30, 'classic', [0, False, False, True, False])
results_3.to_csv('discard_resp.csv', index=False)

100%|██████████| 150/150 [00:09<00:00, 15.90it/s]


In [22]:

results_4 = run_clusterization(no_weighting_corelevance_0_30, 'classic', [0, False, False, False, True])
results_4.to_csv('discard_req.csv', index=False)

100%|██████████| 150/150 [00:09<00:00, 16.30it/s]


### 3.

In [7]:
weighted_corelevance_0_30 = np.load('weighted_corelevance_matrix_0_30.npy')
results_3 = run_clusterization(weighted_corelevance_0_30, 'updated_matrix', '')
results_3.to_csv('mpv_3.csv', index=False)

100%|██████████| 150/150 [00:03<00:00, 45.71it/s]
